# 实验二：单卡 NPU 启动与 DDP 扩展说明

当前实验资源是 `1*NPU 910B3 + 16 vCPU + 32GiB`，因此本章主线是**单卡 NPU 训练启动**。HCCL/DDP 的概念保留为扩展知识，等有多卡或多机资源后再验证。

先把概念讲清楚：

<table style="margin-left: 0; margin-right: auto; text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">概念</th>
      <th style="text-align: left;">第一次出现时的解释</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;">单卡训练</td>
      <td style="text-align: left;">一个训练进程只使用一张 NPU，本实验实际运行 <code>npu:0</code></td>
    </tr>
    <tr>
      <td style="text-align: left;">HCCL</td>
      <td style="text-align: left;">Huawei Collective Communication Library，华为的集合通信库，多卡同步梯度时使用</td>
    </tr>
    <tr>
      <td style="text-align: left;">DDP</td>
      <td style="text-align: left;">DistributedDataParallel，PyTorch 的分布式数据并行封装，每张卡一个进程</td>
    </tr>
    <tr>
      <td style="text-align: left;">rank</td>
      <td style="text-align: left;">分布式训练中的进程编号，单卡时 <code>rank=0</code></td>
    </tr>
    <tr>
      <td style="text-align: left;">world size</td>
      <td style="text-align: left;">参与训练的进程总数，单卡时 <code>world_size=1</code></td>
    </tr>
  </tbody>
</table>


## 当前可运行与暂不运行的内容

<table style="margin-left: 0; margin-right: auto; text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">内容</th>
      <th style="text-align: left;">当前状态</th>
      <th style="text-align: left;">处理方式</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;">单卡 NPU 训练</td>
      <td style="text-align: left;">可以运行</td>
      <td style="text-align: left;">本实验主线</td>
    </tr>
    <tr>
      <td style="text-align: left;">AMP 混合精度</td>
      <td style="text-align: left;">可以运行</td>
      <td style="text-align: left;">后续章节调优</td>
    </tr>
    <tr>
      <td style="text-align: left;">MSPROF profiling</td>
      <td style="text-align: left;">可以运行</td>
      <td style="text-align: left;">后续章节分析瓶颈</td>
    </tr>
    <tr>
      <td style="text-align: left;">HCCL 多卡通信</td>
      <td style="text-align: left;">当前无法真实验证</td>
      <td style="text-align: left;">只讲概念，不作为运行路径</td>
    </tr>
    <tr>
      <td style="text-align: left;">DDP 扩展效率</td>
      <td style="text-align: left;">当前无法真实测量</td>
      <td style="text-align: left;">留到多卡资源后补测</td>
    </tr>
  </tbody>
</table>


## 单卡启动命令

准备好 PASCAL VOC YOLO label 后，在本实验目录执行：

```bash
bash src/scripts/launch_1npu.sh
```

这个脚本做了几件事：

1. 进入实验目录，避免路径混乱。
2. 如果存在 `/usr/local/Ascend/ascend-toolkit/set_env.sh`，就加载 CANN 环境变量。
3. 设置几个 Ascend 运行时环境变量。
4. 调用 `src/scripts/train_yolo_single_npu_amp.py` 启动训练。


In [ ]:
# ====== 1. 查看单卡启动脚本 ======
from pathlib import Path

path = Path('src/scripts/launch_1npu.sh')
print(path.read_text(encoding='utf-8'))


## 启动脚本中的环境变量

第一次看到这些环境变量时，不需要背下来，但要知道它们控制的是 Ascend 运行时行为。

<table style="margin-left: 0; margin-right: auto; text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">环境变量</th>
      <th style="text-align: left;">作用</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;"><code>ASCEND_GLOBAL_LOG_LEVEL</code></td>
      <td style="text-align: left;">控制 Ascend 运行日志级别，数字越小日志越详细</td>
    </tr>
    <tr>
      <td style="text-align: left;"><code>TASK_QUEUE_ENABLE</code></td>
      <td style="text-align: left;">启用任务队列，减少 Host 侧同步等待</td>
    </tr>
    <tr>
      <td style="text-align: left;"><code>COMBINED_ENABLE</code></td>
      <td style="text-align: left;">启用部分运行时融合优化</td>
    </tr>
  </tbody>
</table>

如果训练出错，先不要急着调这些变量。优先检查：NPU 是否可见、torch-npu 是否能 import、VOC 数据路径是否正确。


In [ ]:
# ====== 2. 检查训练入口参数 ======
from pathlib import Path
import yaml

cfg = yaml.safe_load(Path('src/configs/yolo_ascend.yaml').read_text(encoding='utf-8'))
print('output_dir:', cfg['project']['output_dir'])
print('batch_size:', cfg['train']['batch_size'])
print('workers:', cfg['train']['workers'])
print('amp:', cfg['train']['amp'])
print('train_list:', cfg['data']['train_list'])


## HCCL/DDP 扩展预留

多卡训练时，典型命令会变成：

```bash
torchrun --nproc_per_node=8 src/scripts/train_yolo_ddp_amp.py --config src/configs/yolo_ascend.yaml
```

这条命令暂时不执行。这里先解释两个参数：

<table style="margin-left: 0; margin-right: auto; text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">参数</th>
      <th style="text-align: left;">含义</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;"><code>torchrun</code></td>
      <td style="text-align: left;">PyTorch 官方分布式启动器，用来拉起多个训练进程</td>
    </tr>
    <tr>
      <td style="text-align: left;"><code>--nproc_per_node=8</code></td>
      <td style="text-align: left;">每台机器启动 8 个进程，通常对应 8 张 NPU</td>
    </tr>
  </tbody>
</table>

从单卡迁移到 DDP 的核心差异是：每张 NPU 一个进程、`DistributedSampler` 切分数据、DDP 在反向传播后自动同步梯度。本实验代码已经保留这些结构，但当前只走 `world_size=1` 的单卡路径。


In [ ]:
# ====== 3. Show multi-card launch scripts ======
from pathlib import Path

for script in ['src/scripts/launch_1node_8p.sh', 'src/scripts/launch_multinode.sh']:
    print(f'\n===== {script} =====')
    print(Path(script).read_text(encoding='utf-8'))


In [ ]:
%%bash
cat <<'EOF'
# Single-node multi-NPU launch. Change NPU_PER_NODE to match your device count.
NPU_PER_NODE=8 bash src/scripts/launch_1node_8p.sh

# Use only part of the NPUs, for example 2 or 4 NPUs.
NPU_PER_NODE=2 bash src/scripts/launch_1node_8p.sh
NPU_PER_NODE=4 bash src/scripts/launch_1node_8p.sh

# Multi-node multi-NPU launch. Example: 2 nodes, 8 NPUs per node.
# Run on rank 0 node:
MASTER_ADDR=<rank0_ip> NNODES=2 NODE_RANK=0 NPU_PER_NODE=8 bash src/scripts/launch_multinode.sh

# Run on rank 1 node:
MASTER_ADDR=<rank0_ip> NNODES=2 NODE_RANK=1 NPU_PER_NODE=8 bash src/scripts/launch_multinode.sh
EOF


## 本章小结

本章把启动路径明确为单卡 NPU。下一章将解析训练脚本的单卡主流程，并说明哪些代码是为以后 DDP 扩展预留的。


## 课后练习

请根据本节实验内容完成以下练习。题型包含单选题、多选题、判断题、填空题、简答题和代码设计题。

1. (单选题) 本实验默认推荐首先运行哪个训练方式？
   - A. 单卡 NPU 训练
   - B. 8 卡 DDP 训练
   - C. CPU-only 训练
   - D. 只运行数据转换

2. (单选题) `launch_1npu.sh` 的核心作用是？
   - A. 删除数据集
   - B. 设置环境并启动单卡训练脚本
   - C. 把 XML 转成 JSON
   - D. 上传 Git LFS 文件

3. (单选题) DDP 中 `WORLD_SIZE` 通常表示什么？
   - A. 图片宽度
   - B. 总进程数或总设备数
   - C. 类别数量
   - D. 学习率

4. (单选题) 单卡训练中最常见的本地 rank 是？
   - A. 0
   - B. 1
   - C. 8
   - D. 不固定且必须大于 1

5. (多选题) 多卡 DDP 扩展时，需要额外关注哪些内容？
   - A. rank/local_rank/world_size
   - B. HCCL 通信配置
   - C. DistributedSampler
   - D. 每卡 batch size 与总 batch size 的关系

6. (多选题) 单卡启动失败时，可以优先检查哪些问题？
   - A. CANN 环境变量是否 source
   - B. torch_npu 是否可导入
   - C. 数据路径是否存在
   - D. PR 标题是否填写

7. (多选题) 下列哪些环境变量或概念和 Ascend 分布式训练关系更密切？
   - A. RANK
   - B. WORLD_SIZE
   - C. LOCAL_RANK
   - D. HCCL

8. (判断题) 单卡训练跑通后，就可以完全忽略多卡 DDP 的概念。

9. (判断题) 在 notebook 中运行 shell 单元格时，每个单元格都应显式设置关键环境变量更稳妥。

10. (填空题) Ascend 多卡通信通常使用 `____` 后端。

11. (填空题) 单卡训练启动脚本一般会把模型放到 `____` 设备上运行。

12. (简答题) 为什么单卡脚本和 DDP 脚本不应该混在一个不可区分的命令里？

13. (简答题) DDP 中为什么常常只让 rank 0 保存 checkpoint？

14. (简答题) 从单卡扩展到多卡时，学习率和 batch size 为什么需要重新考虑？

15. (代码设计题) 写一段 shell 命令，展示单卡训练前如何设置 CANN 环境并启动 `launch_1npu.sh`。

> 参考答案见 answer/03.03_single_npu_launch_and_ddp_extension_answer.ipynb。
